# Day 024 — Exercise 5: batch_scrape_extract

**What you'll build:** `batch_scrape_extract(urls, fields, model)` — runs `scrape_and_extract` over a list of URLs, catches errors per-URL, and returns a list of `{url, status, data/error}` envelopes.

**Why it matters:** Real pipelines process many pages. One bad URL must not crash the entire batch. This is the Day 22 error-envelope pattern applied to AI scraping — safe, iterable, and inspectable.

In [ ]:
import re
import json
import requests
import ollama
from bs4 import BeautifulSoup

## Provided Helpers

In [ ]:
def clean_html_text(html_string: str) -> str:
    soup = BeautifulSoup(html_string, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_schema_fields(
    text: str, fields: list[str], model: str = "llama3.2"
) -> dict:
    fields_json = json.dumps(fields)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a structured data extractor. "
                    f"Extract the following fields from the text: {fields_json}. "
                    "Return JSON with exactly these keys. "
                    "Use null for any field you cannot find. "
                    "Return only valid JSON, no explanation."
                ),
            },
            {
                "role": "user",
                "content": f"Extract from this text:\n\n{text[:3000]}",
            },
        ],
        format="json",
    )
    raw = response["message"]["content"]
    try:
        return json.loads(raw)
    except Exception:
        return {f: None for f in fields}


def scrape_and_extract(
    url: str, fields: list[str], model: str = "llama3.2"
) -> dict:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    text = clean_html_text(response.text)
    return extract_schema_fields(text, fields, model=model)

## Your Implementation

In [ ]:
def batch_scrape_extract(
    urls: list[str],
    fields: list[str],
    model: str = "llama3.2",
) -> list[dict]:
    """
    Scrape and extract from a list of URLs. Never raises.

    Args:
        urls:   List of URLs to scrape.
        fields: Field names to extract from each page.
        model:  Ollama model name.

    Returns:
        List of result dicts:
          On success: {'url': url, 'status': 'ok',    'data': dict}
          On error:   {'url': url, 'status': 'error', 'error': str}
    """
    # TODO: results = []
    # TODO: for url in urls:
    #           try:
    #               data = scrape_and_extract(url, fields, model=model)
    #               results.append({'url': url, 'status': 'ok', 'data': data})
    #           except Exception as e:
    #               results.append({'url': url, 'status': 'error', 'error': str(e)})
    # TODO: return results
    pass

## Check Your Work

In [ ]:
GOOD_URL = "https://books.toscrape.com"
BAD_URL  = 'http://localhost:9999/'
FIELDS   = ['site_name']


def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'batch_scrape_extract' in globals()
        passed += 1; print('\u2705 Check 1: batch_scrape_extract defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    results = None

    # Check 2: returns a list of 2 results (1 net + 1 LLM call total)
    try:
        results = batch_scrape_extract([GOOD_URL, BAD_URL], FIELDS)
        assert isinstance(results, list), f'expected list, got {type(results)}'
        assert len(results) == 2, f'expected 2 results, got {len(results)}'
        passed += 1; print('\u2705 Check 2: returns a list with 2 results')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: good URL → status='ok'
    try:
        assert results is not None, 'results is None (Check 2 failed)'
        ok = results[0]
        assert ok['status'] == 'ok', \
            f"expected status='ok' for good URL, got {ok['status']!r}"
        assert 'data' in ok, f"'data' key missing from ok result: {ok}"
        passed += 1; print("\u2705 Check 3: good URL has status='ok' with data")
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: bad URL → status='error'
    try:
        assert results is not None, 'results is None'
        err = results[1]
        assert err['status'] == 'error', \
            f"expected status='error' for bad URL, got {err['status']!r}"
        assert 'error' in err, f"'error' key missing from error result: {err}"
        passed += 1; print("\u2705 Check 4: bad URL has status='error' with error message")
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: each result has a 'url' key matching the input
    try:
        assert results is not None, 'results is None'
        assert results[0]['url'] == GOOD_URL, \
            f"url mismatch: expected {GOOD_URL!r}, got {results[0]['url']!r}"
        assert results[1]['url'] == BAD_URL, \
            f"url mismatch: expected {BAD_URL!r}, got {results[1]['url']!r}"
        passed += 1; print('\u2705 Check 5: each result has correct url key')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def batch_scrape_extract(
    urls: list[str],
    fields: list[str],
    model: str = "llama3.2",
) -> list[dict]:
    results = []
    for url in urls:
        try:
            data = scrape_and_extract(url, fields, model=model)
            results.append({"url": url, "status": "ok", "data": data})
        except Exception as e:
            results.append({"url": url, "status": "error", "error": str(e)})
    return results
```

</details>